# Testing Celery Set-Up: Submitting jobs to workers

Create and run a Celery worker (see example in AISynbioPipeline/aisynbiopipeline/tasks/fastp_task.py)
Create a celery app/worker called fastp that talks to defined broker (redis)
Configure the celery app/worker
Define what the worker can do (function, inputs, outputs)
Instantiate the worker
Run worker/ Bring worker online: Now, must run this script in order to make the worker available to receive jobs. This also makes the worker available to anybody else who has access to the celery client.
Send job to worker: To utilize this worker (e.g. in Jupyter), create the celery client, send tasks to celery client.

## Set-up

In [ ]:
## Make sure running in aisynbio_env

In [498]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [72]:
import sys
import os

# Add project root to path for access to workflows and tasks
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [73]:
import os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

## Create mock seq order

In [6]:
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample

# Create mock SeqOrder for testing
mock_order = SeqOrder('mock_SeqOrder_12-04-25', create=True)

# Define 2 samples to copy into mock SeqOrder
library = Library('ANL_ALE1b', 'Illumina')
sample1 = SeqSample(library, 'E2')
sample2 = SeqSample(library, 'F2')

# Copy samples into mock SeqOrder and delete anything but raw data files
sample1_cp = sample1.copy(mock_order)
sample2_cp = sample2.copy(mock_order)
sample1_cp.delete('trimmed')
sample2_cp.delete('trimmed')


All associated files and folders of SeqSample E2 were copied to /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25. Relative file paths were preserved.
All associated files and folders of SeqSample F2 were copied to /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25. Relative file paths were preserved.


## Run fastp - directly and as Celery task

In [8]:
# Run fastp on sample1_cp (no Celery)

from aisynbiopipeline.workflows.read_qc import run_fastp

fwd_in_path = sample1_cp.received[0]
fwd_in_file = os.path.basename(fwd_in_path)
fwd_out_file = fwd_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
fwd_out_path = os.path.join(mock_library.path, 'trimmed', fwd_out_file)
rvs_in_path = sample1_cp.received[1]
rvs_in_file = os.path.basename(rvs_in_path)
rvs_out_file = rvs_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
rvs_out_path = os.path.join(mock_library.path, 'trimmed', rvs_out_file)

In [11]:
run_fastp(fwd_in_path, rvs_in_path, fwd_out_path, rvs_out_path, threads=16, polyG=5)

Running fastp on /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/received/E2_S1_L001_R1_001.fastq.gz and /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/received/E2_S1_L001_R2_001.fastq.gz


Read1 before filtering:
total reads: 1117271
total bases: 280435021
Q20 bases: 236208701(84.2294%)
Q30 bases: 224106189(79.9138%)
Q40 bases: 493663(0.176035%)

Read2 before filtering:
total reads: 1117271
total bases: 280435021
Q20 bases: 224169995(79.9365%)
Q30 bases: 208191250(74.2387%)
Q40 bases: 297544(0.106101%)

Read1 after filtering:
total reads: 1111607
total bases: 153131164
Q20 bases: 151413497(98.8783%)
Q30 bases: 150829213(98.4967%)
Q40 bases: 484785(0.316582%)

Read2 after filtering:
total reads: 1111607
total bases: 153131164
Q20 bases: 149879560(97.8766%)
Q30 bases: 148874747(97.2204%)
Q40 bases: 292818(0.19122%)

Filtering result:
reads passed filter: 2223214
reads failed due to low quality: 11288
reads failed due to too many N: 40
reads failed due to too short: 0
reads with adapter trimmed: 2031060
bases trimmed due to adapters: 251773874

Duplication rate: 0.0183483%

Insert size peak (evaluated by paired-end reads): 105

JSON report: /storage/synbio/ai_synbio_data/ex

In [12]:
sample1_cp.trimmed

[PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/E2_S1_L001_R1_001_trimmed.fastq.gz'),
 PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/E2_S1_L001_R2_001_trimmed.fastq.gz')]

In [44]:
# Run fastp on sample2_cp as Celery task

from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)


In [105]:
from aisynbiopipeline.workflows.read_qc import run_fastp
from pathlib import Path

fwd_in_path = sample2_cp.received[0]
fwd_in_file = os.path.basename(fwd_in_path)
fwd_out_file = fwd_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
fwd_out_path = os.path.join(mock_library.path, 'trimmed', fwd_out_file)
rvs_in_path = sample2_cp.received[1]
rvs_in_file = os.path.basename(rvs_in_path)
rvs_out_file = rvs_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
rvs_out_path = os.path.join(mock_library.path, 'trimmed', rvs_out_file)

# Normalize Path → str - Celery tasks only accept certain input data types.
def norm(x): return str(x) if isinstance(x, Path) else x

fastp_params = {
    'path_to_fwd': norm(fwd_in_path),
    'path_to_rev': norm(rvs_in_path),
    'path_to_fwd_out': norm(fwd_out_path),
    'path_to_rev_out': norm(rvs_out_path),
    'threads': 16,
    'polyG':5
}

# Submit task

result = client.send_task(
    'fastp.run',
    kwargs=fastp_params
)

In [106]:
result.state

'FAILURE'

In [103]:
result.get()

NotRegistered: 'fastp.run'

In [63]:
sample2_cp.trimmed

[PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/F2_S7_L001_R1_001_trimmed.fastq.gz'),
 PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/F2_S7_L001_R2_001_trimmed.fastq.gz')]

In [64]:
sample2_cp.breseq

PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/breseq/F2')

## Run breseq - directly and as Celery task

In [83]:
# Run breseq on sample_cp1 (No celery)

from aisynbiopipeline.workflows.breseq import Breseq, Breseq_params

params = Breseq_params('ADP1_Neidle_CDM.gbk', True, limit_fold_coverage=100, num_processors=4)
breseq = Breseq(sample1_cp.trimmed, params, sample1_cp.breseq)
print(breseq.params.version_name)
breseq.run()

breseq_28ebb29ed2


In [117]:
# Run breseq on sample2_cp as Celery task

from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

In [158]:
# Submit task

# Normalize Path → str - Celery tasks only accept certain input data types.
def norm(x): return str(x) if isinstance(x, Path) else x

breseq_params = {
    'read_paths': [norm(x) for x in sample2_cp.trimmed],
    'breseq_folder': norm(sample2_cp.breseq),
    'reference': 'ADP1_Neidle_CDM.gbk',
    'polymorphism_prediction': True,
    'limit_fold_coverage': 100,
    'num_processors': 4
}

result = client.send_task(
    'breseq.run',
    kwargs=breseq_params
)

In [161]:
result.state

ValueError: Exception information must include the exception type

In [157]:
result.get()

{'status': 'success', 'output': 'test'}

In [183]:
a = Breseq.from_existing(os.path.join(sample1_cp.breseq, 'breseq_28ebb29ed2'))
a.breseq_run_time_minutes

22

In [182]:
b = Breseq.from_existing(os.path.join(sample2_cp.breseq, 'breseq_28ebb29ed2'))
b.breseq_run_time_minutes

14

## Run Celery tasks in parallel

In [204]:
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample

# Define mock SeqOrder for testing
mock_order = SeqOrder('mock_SeqOrder_12-04-25')

# Identify two more samples to copy into mock SeqOrder to test parallel task execution
library = Library('ANL_ALE1b', 'Illumina')
sample3 = SeqSample(library, 'E4')
sample4 = SeqSample(library, 'F4')

# Copy samples into mock SeqOrder and delete anything but raw data files
sample3_cp = sample3.copy(mock_order)
sample3_cp.delete('trimmed')
sample4_cp = sample4.copy(mock_order)
sample4_cp.delete('trimmed')


All associated files and folders of SeqSample F4 were copied to /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25. Relative file paths were preserved.


In [393]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)


In [398]:
# Run fastp on sample3_cp and sample4_cp as Celery task in parallel

from pathlib import Path

seqsamples = [sample1_cp, sample2_cp, sample3_cp, sample4_cp]

def get_fastp_params(seqsample):

    fwd_in_path = seqsample.received[0]
    fwd_in_file = os.path.basename(fwd_in_path)
    fwd_out_file = fwd_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    fwd_out_path = os.path.join(mock_library.path, 'trimmed', fwd_out_file)
    rvs_in_path = seqsample.received[1]
    rvs_in_file = os.path.basename(rvs_in_path)
    rvs_out_file = rvs_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    rvs_out_path = os.path.join(mock_library.path, 'trimmed', rvs_out_file)
    
    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    fastp_params = {
        'path_to_fwd': norm(fwd_in_path),
        'path_to_rev': norm(rvs_in_path),
        'path_to_fwd_out': norm(fwd_out_path),
        'path_to_rev_out': norm(rvs_out_path),
        'threads': 16,
        'polyG':5
    }

    return fastp_params

# Submit task

results = []

for sample in seqsamples:
    result = client.send_task(
        'fastp.run',
        kwargs=get_fastp_params(sample),
        queue='fastp'
    )
    results.append(result)

In [425]:
for i in range(4):
    print(results[i].status)

SUCCESS
SUCCESS
SUCCESS
SUCCESS


In [426]:
for i in range(4):
    print(results[i].id)
    print(results[i].status)
    print(results[i].traceback)
    print(results[i].result)

5e2ef5dc-a15b-4980-ad9b-415b7908949a
SUCCESS
None
{'status': 'success', 'output': {'out_R1': '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/E2_S1_L001_R1_001_trimmed.fastq.gz', 'out_R2': '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/E2_S1_L001_R2_001_trimmed.fastq.gz'}}
ebdf5a06-8df0-4d29-a989-a5d4f6f5c3de
SUCCESS
None
{'status': 'success', 'output': {'out_R1': '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/F2_S7_L001_R1_001_trimmed.fastq.gz', 'out_R2': '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/F2_S7_L001_R2_001_trimmed.fastq.gz'}}
60c8c2cf-92cd-494f-9ce6-9c719649ec53
SUCCESS
None
{'status': 'success', 'output': {'out_R1': '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrd

In [420]:
# sample1_cp.delete('trimmed')
# sample2_cp.delete('trimmed')
# sample3_cp.delete('trimmed')
# sample4_cp.delete('trimmed')

In [428]:
print(sample1_cp.trimmed)
print(sample2_cp.trimmed)
print(sample3_cp.trimmed)
print(sample4_cp.trimmed)

[PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/E2_S1_L001_R1_001_trimmed.fastq.gz'), PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/E2_S1_L001_R2_001_trimmed.fastq.gz')]
[PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/F2_S7_L001_R1_001_trimmed.fastq.gz'), PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/F2_S7_L001_R2_001_trimmed.fastq.gz')]
[PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/E4_S2_L001_R1_001_trimmed.fastq.gz'), PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/E4_S2_L001_R2_001_trimmed.fastq.gz')]
[

In [483]:
sample1_cp.delete('breseq')
sample2_cp.delete('breseq')
sample3_cp.delete('breseq')
sample4_cp.delete('breseq')

In [484]:
# Run breseq on all samples as Celery tasks in parallel

from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

In [486]:
seqsamples = [sample1_cp, sample2_cp, sample3_cp, sample4_cp]

def get_breseq_params(seqsample):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': 'ADP1_Neidle_CDM.gbk',
        'polymorphism_prediction': True,
        'limit_fold_coverage': 100,
        'num_processors': 4
    }
    return breseq_params

result = client.send_task(
    'breseq.run',
    kwargs=breseq_params
)

# Submit tasks

results = []
for sample in seqsamples:
    result = client.send_task(
        'breseq.run',
        kwargs=get_breseq_params(sample),
        queue='breseq'
    )
    results.append(result)

In [493]:
for i in range(4):
    print(results[i].status)

SUCCESS
SUCCESS
SUCCESS
SUCCESS


In [490]:
for i in range(4):
    print(results[i].id)
    print(results[i].status)
    print(results[i].traceback)
    print(results[i].result)

9985ac08-0765-4f39-9edc-c457a93ad70f
STARTED
None
{'read_paths': ['/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/E2_S1_L001_R1_001_trimmed.fastq.gz', '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/E2_S1_L001_R2_001_trimmed.fastq.gz'], 'breseq_folder': '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/breseq/E2', 'reference': 'ADP1_Neidle_CDM.gbk', 'polymorphism_prediction': True, 'limit_fold_coverage': 100, 'num_processors': 4}
69e027b2-69e5-4221-bd36-c7d300328f4a
STARTED
None
{'read_paths': ['/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/F2_S7_L001_R1_001_trimmed.fastq.gz', '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mock_SeqOrder_12-04-25/ANL_ALE1b_Illumina/trimmed/F2_S7_L001_R2_001_trimmed.f

In [527]:
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders

list_seqorders()

['Plasmidsaurus_9-13-2025_HTGS8F',
 'Plasmidsaurus_10-06-2025_P4CYGL',
 'ANL_ALE1b',
 'mock_SeqOrder_12-04-25',
 'Plasmidsaurus_11-26-2025_P4CYGL']

In [540]:
from aisynbiopipeline.workflows.plasmidsaurus import get_access_token, download_results, get_credentials

p4 = SeqOrder('Plasmidsaurus_11-26-2025_P4CYGL', create=True)
item_code = 'P4CYGL'
CLIENT_ID = get_credentials("PLASMIDSAURUS_CLIENT_ID")
CLIENT_SECRET = get_credentials("PLASMIDSAURUS_CLIENT_SECRET")
access_token = get_access_token(CLIENT_ID, CLIENT_SECRET)
seqorder_dir = p4.path
download_results(item_code, access_token, seqorder_dir)

ITEM P4CYGL
{'code': 'P4CYGL',
 'done_date': '2025-09-30T21:37:37.602680+00:00',
 'gross': 540.0,
 'order_name': '',
 'product_name': 'hybrid_extraction',
 'quantity': 3,
 'status': 'complete'}



DOWNLOADING RESULTS FOR P4CYGL 

[                                                  ] 3070976/777005954 bytes

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[=                                                 ] 17762304/777005954 bytes

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==                                                ] 32607232/777005954 bytes

...

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[================================================= ] 10897160192/10922136720 bytes

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[================================================= ] 10911591424/10922136720 bytes

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [561]:
os.listdir(os.path.join(p4.path))

['P4CYGL_results.zip',
 'P4CYGL_results',
 'P4CYGL_reads.zip',
 'P4CYGL_reads',
 'Plasmidsaurus_11-26-2025_P4CYGL_Illumina']

In [546]:
os.listdir(os.path.join(p4.path, 'P4CYGL_results', 'P4CYGL_1_ANLstock.ACN2586.colony1'))

['ONT-plus-Illumina-polished', 'ONT-only']

In [543]:
os.listdir(os.path.join(p4.path, 'P4CYGL_reads'))

['P4CYGL_3_ANLstock.ACN2586.colony3_nanopore.fastq.gz',
 'P4CYGL_3_ANLstock.ACN2586.colony3_illumina_R1.fastq.gz',
 'P4CYGL_3_ANLstock.ACN2586.colony3_illumina_R2.fastq.gz',
 'P4CYGL_1_ANLstock.ACN2586.colony1_nanopore.fastq.gz',
 'P4CYGL_1_ANLstock.ACN2586.colony1_illumina_R1.fastq.gz',
 'P4CYGL_1_ANLstock.ACN2586.colony1_illumina_R2.fastq.gz',
 'P4CYGL_2_ANLstock.ACN2586.colony2_nanopore.fastq.gz',
 'P4CYGL_2_ANLstock.ACN2586.colony2_illumina_R1.fastq.gz',
 'P4CYGL_2_ANLstock.ACN2586.colony2_illumina_R2.fastq.gz']

In [580]:
p4.libraries

[]

In [583]:
short = Library(p4, 'Illumina', create=True)
long = Library(p4, 'Nanopore', create=True)

In [584]:
p4.libraries

[PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_11-26-2025_P4CYGL/Plasmidsaurus_11-26-2025_P4CYGL_Illumina'),
 PosixPath('/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_11-26-2025_P4CYGL/Plasmidsaurus_11-26-2025_P4CYGL_Nanopore')]

In [597]:
def create_seqorder_name(item_code):

    client_id = get_credentials("PLASMIDSAURUS_CLIENT_ID")
    client_secret = get_credentials("PLASMIDSAURUS_CLIENT_SECRET")
    access_token = get_access_token(client_id, client_secret)
    res = requests.get(
        f"{API_URL}/api/item/{item_code}",
        headers={"Authorization": f"Bearer {access_token}"},
    )
    prefix = 'Plasmidsaurus'
    date = res.json()['done_date'].split('T')[0]
    code = res.json()['code']
    
    seqorder_name = ('_').join([prefix, date, code])
    
    return seqorder_name

'Plasmidsaurus_2025-09-30_P4CYGL'

In [600]:
from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name

create_seqorder_name('P4CYGL')

'Plasmidsaurus_2025-09-30_P4CYGL'